#### Is it possible to tell whether a specific kind of textile (protective workwear, bed linen etc) is produced in specific countries mostly?

In [4]:
import os
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from rapidfuzz import process, fuzz

# ==========================================
# SYSTEM WORKSPACE INTEGRATION PATHS
# ==========================================

# 1. Step backward out of 'notebooks/' to the root directory
root_dir = os.path.abspath(os.path.join('..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# 2. Import your central configuration framework module
import config

# ==========================================
# 1) ADJUST ROBUST PATHS BASED ON YOUR FILES
# ==========================================

# Switch extensions here if your files are .xlsx or .csv on disk:
ted_file_name = "ted_only_selected_notices.xlsx"  # Change to .csv if needed
trade_file_name = "merged_tradeatlas_clean.xlsx"

ted_file_path = os.path.join(config.OUTPUT_DIR, ted_file_name)
trade_file_path = os.path.join(config.OUTPUT_DIR, trade_file_name)


# ==========================================
# 2) LOAD DATASETS SAFELY
# ==========================================

print(f"🔄 Loading TED data from: {ted_file_path}")
if ted_file_name.endswith(".xlsx"):
    df_ted = pd.read_excel(ted_file_path)
else:
    df_ted = pd.read_csv(ted_file_path)

print(f"🔄 Loading TradeAtlas data from: {trade_file_path}")
df_trade = pd.read_excel(trade_file_path)


# ==========================================
# 3) DATA FILTERING & CLEANING PREPROCESS
# ==========================================

mask_multi = df_ted["winner_name"].str.contains(r",\s*", na=False)
df_multinames = df_ted[mask_multi]
df_ted = df_ted[~mask_multi]

# Slice down structural matrices
df_ted = df_ted[
    ["winner_name", "winner_country", "buyer_country", "buyer_town",
     "buyer_type", "publication_date", "main_classification"]
].dropna(subset=["winner_name", "publication_date"])

df_trade = df_trade[
    ["importer_name", "exporter_name", "exporter_country",
     "usd_fob", "arrival_date", "hs_code"]
].dropna(subset=["importer_name", "arrival_date", "hs_code"])

# Enforce clean datetime objects
df_ted["publication_date"] = pd.to_datetime(df_ted["publication_date"], errors="coerce")
df_trade["arrival_date"] = pd.to_datetime(df_trade["arrival_date"], errors="coerce")

print("✅ Success! Both datasets loaded and preprocessed smoothly.")

🔄 Loading TED data from: c:\Users\aylin\Python_projects\ted-trade-pipeline\outputs\ted_only_selected_notices.xlsx
🔄 Loading TradeAtlas data from: c:\Users\aylin\Python_projects\ted-trade-pipeline\outputs\merged_tradeatlas_clean.xlsx
✅ Success! Both datasets loaded and preprocessed smoothly.


In [5]:
def preprocess_name(name):
    if pd.isna(name):
        return ""
    clean = (
        str(name).lower()
        .replace("&", "and")
        .replace(",", "")
        .replace(".", "")
        .replace("gmbh & co kg", "gmbh")
        .replace("co kg", "")
        .strip()
    )
    return " ".join(clean.split()[:3])

df_ted["short_winner"] = df_ted["winner_name"].apply(preprocess_name)
df_trade["short_importer"] = df_trade["importer_name"].apply(preprocess_name)


# 2) SINGLE CLEANING FUNCTION (used everywhere)

def clean_company_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()

    suffixes = [
        r"gmbh", r"gmbh & co\.? kg", r"& co\.? kg", r"co\.? kg",
        r"kg", r"ag", r"se", r"ohg", r"ltd\.?", r"llc",
        r"inc\.?", r"e\.k\.?", r"gmbh & co", r"gmbh \+ co\.? kg",
        r"\+ co\.? kg"
    ]

    pattern = "(" + "|".join(suffixes) + ")"
    name = re.sub(pattern, "", name).strip()

    # remove punctuation / normalize spacing
    name = re.sub(r"[^\w\s\-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Normalize names ONCE
df_ted["short_winner"] = df_ted["winner_name"].apply(clean_company_name)
df_trade["short_importer"] = df_trade["importer_name"].apply(clean_company_name)


# 3) Multi-strategy fuzzy matching (but simple)

def multi_strategy_match(ted_name, trade_names):

    # simple variants built from ONE normalization function
    variants = [
        ted_name,
        clean_company_name(ted_name),
        " ".join(ted_name.split()[:3]),  # first 3 words
        " ".join(ted_name.split()[:2])   # first 2 words
    ]

    best_match = None
    best_score = 0

    for v in variants:
        match = process.extractOne(v, trade_names, scorer=fuzz.token_set_ratio)
        if match and match[1] > best_score:
            best_match = match[0]
            best_score = match[1]

    return (best_match, best_score) if best_score >= 85 else (None, None)


# 4) FINAL MATCHING FUNCTION (clean + minimal)

def match_by_name(df_ted, df_trade, similarity_threshold=85):

    results = []
    trade_names = df_trade["short_importer"].unique()

    for _, ted_row in df_ted.iterrows():

        name_clean = ted_row["short_winner"]
        if not name_clean:
            continue

        importer_match, score = multi_strategy_match(name_clean, trade_names)

        if importer_match and score >= similarity_threshold:

            matched_rows = df_trade[df_trade["short_importer"] == importer_match]

            for _, trade_row in matched_rows.iterrows():
                results.append({
                    "winner_name": ted_row["winner_name"],
                    "short_winner": ted_row["short_winner"],
                    "winner_country": ted_row["winner_country"],
                    "buyer_country": ted_row["buyer_country"],
                    "buyer_town": ted_row["buyer_town"],
                    "buyer_type": ted_row["buyer_type"],
                    "publication_date": ted_row["publication_date"],
                    "main_classification": ted_row["main_classification"],
                    "importer_name": trade_row["importer_name"],
                    "short_importer": trade_row["short_importer"],
                    "exporter_name": trade_row["exporter_name"],
                    "exporter_country": trade_row["exporter_country"],
                    "hs_code": trade_row["hs_code"],
                    "usd_fob": trade_row["usd_fob"],
                    "arrival_date": trade_row["arrival_date"],
                    "similarity": score
                })

    df_matches = pd.DataFrame(results)

    if not df_matches.empty:
        df_matches["days_difference"] = (
            df_matches["arrival_date"] - df_matches["publication_date"]
        ).abs().dt.days

        df_matches["months_difference"] = (df_matches["days_difference"] / 30).round(1)

    return df_matches


# 5) RUN MATCHING

df_all_matches = match_by_name(df_ted, df_trade)

print("Matched winners:", df_all_matches["winner_name"].nunique())
df_all_matches.head()


Matched winners: 85


,winner_name,short_winner,winner_country,buyer_country,buyer_town,buyer_type,publication_date,main_classification,importer_name,short_importer,exporter_name,exporter_country,hs_code,usd_fob,arrival_date,similarity,days_difference,months_difference
0,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.0,2020-02-02,95.652174,1088,36.3
1,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,68159990,470.0,2020-02-02,95.652174,1088,36.3
2,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.0,2020-02-02,95.652174,1088,36.3
3,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,68159990,470.0,2020-02-02,95.652174,1088,36.3
4,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.0,2020-02-02,95.652174,1088,36.3


In [6]:
df_all_matches[["winner_name", "importer_name", "similarity"]].drop_duplicates().to_csv("outputs/matched_winners.csv")

In [7]:
df_all_matches.to_csv("outputs/df_all_matches.csv")

In [8]:
df_all_matches.winner_name.nunique()

85

In [9]:
df_all_matches

,winner_name,short_winner,winner_country,buyer_country,buyer_town,buyer_type,publication_date,main_classification,importer_name,short_importer,exporter_name,exporter_country,hs_code,usd_fob,arrival_date,similarity,days_difference,months_difference
0,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.00,2020-02-02,95.652174,1088,36.3
1,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,68159990,470.00,2020-02-02,95.652174,1088,36.3
2,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.00,2020-02-02,95.652174,1088,36.3
3,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,68159990,470.00,2020-02-02,95.652174,1088,36.3
4,Detlef Paulsen BSA GmbH,detlef pauln bsa,DE,DE,Kiel,Police,2023-01-25,18000000,DETLEF PAUL,detlef paul,OSWAL TRADERS & TRAVELS PVT. LTD.,India,44209090,15.00,2020-02-02,95.652174,1088,36.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87651,Ziegler Textil GmbH,ziegler textil,Germany,Germany,NaN,cities,2024-01-31,Weatherproof clothing,ZIEGLER TEXTIL GMBH,ziegler textil,AMRIT EXPORTS PVT. LTD.,India,62034990,4372.55,2018-07-25,100.000000,2016,67.2
87652,Ziegler Textil GmbH,ziegler textil,Germany,Germany,NaN,cities,2024-01-31,Weatherproof clothing,ZIEGLER TEXTIL GMBH,ziegler textil,AMRIT EXPORTS PVT. LTD.,India,62034990,2456.50,2018-07-25,100.000000,2016,67.2
87653,Ziegler Textil GmbH,ziegler textil,Germany,Germany,NaN,cities,2024-01-31,Weatherproof clothing,ZIEGLER TEXTIL GMBH,ziegler textil,AMRIT EXPORTS PVT. LTD.,India,62034990,6035.00,2018-07-12,100.000000,2029,67.6
87654,Ziegler Textil GmbH,ziegler textil,Germany,Germany,NaN,cities,2024-01-31,Weatherproof clothing,ZIEGLER TEXTIL GMBH,ziegler textil,AMRIT EXPORTS PVT. LTD.,India,62033990,6844.50,2018-01-24,100.000000,2198,73.3


In [10]:
pubdate_counts = (
    df_all_matches
    .groupby("winner_name")["publication_date"]
    .nunique()
    .reset_index(name="unique_publication_dates")
)


In [11]:
pubdate_counts.winner_name.unique()

<StringArray>
[                                                'AKAH - Albrecht Kind GmbH',
                                                  'Adolf Würth GmbH & Co.KG',
                                                        'Albrecht Kind GmbH',
                                      'Arbeitsschutz RheinRuhr Service GmbH',
                                            'Becon Berliner Confection GmbH',
                                                   'Beirholms VÃ¦verier A/S',
                                                            'Berendsen GmbH',
                                            'Bierbaum-Proenen GmbH & Co. KG',
                                                   'Bonowi Hart Armour GmbH',
                                        'Brügelmann Textilien GmbH & Co. KG',
                                                        'COP Vertriebs-GmbH',
                                    'CWS Workwear Deutschland GmbH & Co. KG',
                                                  

In [12]:
# Filter by time window
def filter_by_time(df_matches, max_days):
    return df_matches[df_matches["days_difference"] <= max_days]

df_6m = filter_by_time(df_all_matches, 180).drop_duplicates()

# --- Textile HS code mapping ---
hs_mapping = {
    "610510": "Men’s cotton shirts",
    "610910": "T-shirts, cotton",
    "611420": "Cotton sweatshirts / pullovers",
    "611610": "Gloves, knitted or crocheted",
    "611692": "Gloves, synthetic fibres",
    "620312": "Men’s suits, synthetic fibres",
    "620319": "Men’s jackets & blazers (other textiles)",
    "620333": "Men’s coats, synthetic fibres",
    "620729": "Blouses, man-made fibres",
    "621010": "Protective garments (industrial / medical)",
    "621133": "Men’s coats, man-made fibres",
    "630231": "Bed linen, cotton",
    "630260": "Toilet and kitchen linen, terry fabric",
    "520852": "Woven cotton fabrics (dyed)",
    "540752": "Woven synthetic fabrics (dyed)",
    "640610": "Protective footwear (textile upper)",
    "420291": "Textile gloves and similar accessories",
    "420329": "Protective clothing accessories, textile",
}

non_textile = ["847141", "854231"]  # irrelevant base codes

# --- Clean code strings ---
df_6m["hs_code"] = df_6m["hs_code"].astype(str).str.strip()

# --- Filter out non-textile HS codes (if they start with irrelevant prefixes) ---
df_6m = df_6m[~df_6m["hs_code"].str.startswith(tuple(non_textile))].copy()

# --- Map textile HS codes by prefix matching ---
def map_hs_description(code):
    if pd.isna(code):
        return None
    code_str = str(code)
    for base, desc in hs_mapping.items():
        if code_str.startswith(base):
            return desc
    return "Other textile articles"

df_6m["description"] = df_6m["hs_code"].apply(map_hs_description)

In [13]:
# df_6m.winner_name.value_counts().to_csv("outputs\matched_winners.csv")

In [14]:
df_6m.winner_name.nunique()

47

In [15]:
import urllib.parse
import pycountry

# Convert "Pakistan" → "PK"
def iso2(country):
    try:
        return pycountry.countries.lookup(country).alpha_2
    except:
        return None   # or "XX" if you prefer

# Extract first two words of exporter name
def first_two(name):
    if not isinstance(name, str):
        return ""
    parts = name.split()
    return " ".join(parts[:2])

# Build OSH link
def make_osh_link(exporter_name, exporter_country):
    code = iso2(exporter_country)
    if code is None:
        return None  # skip unknown countries
    
    q = urllib.parse.quote_plus(first_two(exporter_name))
    return f"https://opensupplyhub.org/facilities?q={q}&countries={code}&sort_by=contributors_desc"

# Apply to your dataframe
df_6m["osh_link"] = df_6m.apply(
    lambda r: make_osh_link(r["exporter_name"], r["exporter_country"]),
    axis=1
)


In [17]:
df_6m["osh_link"].drop_duplicates().to_csv("outputs/osh_links.csv")

In [18]:
df_6m["short_exporter_name"] = df_6m["exporter_name"].apply(clean_company_name)


In [20]:
df_6m.short_exporter_name.drop_duplicates().to_csv("outputs/unique_exporter.csv")

In [21]:
df_6m

,winner_name,short_winner,winner_country,buyer_country,buyer_town,buyer_type,publication_date,main_classification,importer_name,short_importer,...,exporter_country,hs_code,usd_fob,arrival_date,similarity,days_difference,months_difference,description,osh_link,short_exporter_name
5932,Mehler Vario System GmbH,mehler vario system,DE,DE,Duisburg,Police,2020-09-28,35200000,MEHLER VARIO SYSTEM GMBH.,mehler vario system,...,Sri Lanka,58064000,464.96,2021-02-10,100.000000,135,4.5,Other textile articles,https://opensupplyhub.org/facilities?q=AQUA+DY...,aqua dynamics pvt
5933,Mehler Vario System GmbH,mehler vario system,DE,DE,Duisburg,Police,2020-09-28,35200000,MEHLER VARIO SYSTEM GMBH.,mehler vario system,...,Sri Lanka,58063200,862.94,2021-02-10,100.000000,135,4.5,Other textile articles,https://opensupplyhub.org/facilities?q=AQUA+DY...,aqua dynamics pvt
5934,Mehler Vario System GmbH,mehler vario system,DE,DE,Duisburg,Police,2020-09-28,35200000,MEHLER VARIO SYSTEM GMBH.,mehler vario system,...,Sri Lanka,39269099,177.18,2021-02-10,100.000000,135,4.5,Other textile articles,https://opensupplyhub.org/facilities?q=AQUA+DY...,aqua dynamics pvt
5935,Mehler Vario System GmbH,mehler vario system,DE,DE,Duisburg,Police,2020-09-28,35200000,MEHLER VARIO SYSTEM GMBH.,mehler vario system,...,Sri Lanka,39209290,143.48,2021-02-10,100.000000,135,4.5,Other textile articles,https://opensupplyhub.org/facilities?q=AQUA+DY...,aqua dynamics pvt
5936,Mehler Vario System GmbH,mehler vario system,DE,DE,Duisburg,Police,2020-09-28,35200000,MEHLER VARIO SYSTEM GMBH.,mehler vario system,...,Sri Lanka,42021900,95530.58,2021-02-10,100.000000,135,4.5,Other textile articles,https://opensupplyhub.org/facilities?q=AQUA+DY...,aqua dynamics pvt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87588,Delta-Vertrieb GmbH,delta-vertrieb,Germany,Germany,NaN,cities,2024-02-02,Protective gear,DELTA VERTRIEB GMBH,delta vertrieb,...,Sri Lanka,64041900,4348.50,2023-10-20,92.857143,105,3.5,Other textile articles,https://opensupplyhub.org/facilities?q=WORKWEA...,workwear lanka pvt
87589,Delta-Vertrieb GmbH,delta-vertrieb,Germany,Germany,NaN,cities,2024-02-02,Protective gear,DELTA VERTRIEB GMBH,delta vertrieb,...,Sri Lanka,64041900,8332.00,2023-09-26,92.857143,129,4.3,Other textile articles,https://opensupplyhub.org/facilities?q=WORKWEA...,workwear lanka pvt
87590,Delta-Vertrieb GmbH,delta-vertrieb,Germany,Germany,NaN,cities,2024-02-02,Protective gear,DELTA VERTRIEB GMBH,delta vertrieb,...,Sri Lanka,64041900,12060.00,2023-09-22,92.857143,133,4.4,Other textile articles,https://opensupplyhub.org/facilities?q=WORKWEA...,workwear lanka pvt
87591,Delta-Vertrieb GmbH,delta-vertrieb,Germany,Germany,NaN,cities,2024-02-02,Protective gear,DELTA VERTRIEB GMBH,delta vertrieb,...,Sri Lanka,61161000,614.16,2023-08-29,92.857143,157,5.2,"Gloves, knitted or crocheted",https://opensupplyhub.org/facilities?q=WORKWEA...,workwear lanka pvt


In [22]:
df_6m.exporter_name

5932      AQUA DYNAMICS PVT LTD.
5933      AQUA DYNAMICS PVT LTD.
5934      AQUA DYNAMICS PVT LTD.
5935      AQUA DYNAMICS PVT LTD.
5936      AQUA DYNAMICS PVT LTD.
                  ...           
87588    WORKWEAR LANKA PVT LTD.
87589    WORKWEAR LANKA PVT LTD.
87590    WORKWEAR LANKA PVT LTD.
87591    WORKWEAR LANKA PVT LTD.
87592    WORKWEAR LANKA PVT LTD.
Name: exporter_name, Length: 4275, dtype: str

In [23]:
df_ted.winner_name.nunique()

165

In [24]:

# Prepare for grouping
df = df_6m.copy()
df["year"] = df["publication_date"].dt.year
df["exporter_country"] = df["exporter_country"].fillna("Unknown")
df["winner_name"] = df["winner_name"].fillna("Unknown")

# Exporter ranking per winner-year
full_exporter_stats = (
    df.groupby(["winner_name", "year", "exporter_country"])
      .size()
      .reset_index(name="num_records")
      .sort_values(["winner_name", "year", "num_records"], ascending=[True, True, False])
)

full_exporter_stats["rank"] = (
    full_exporter_stats.groupby(["winner_name", "year"])["num_records"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# --- Aggregate to winner-level TOTAL country counts ---
winner_country_totals = (
    full_exporter_stats.groupby(["winner_name", "exporter_country"])["num_records"]
    .sum()
    .reset_index()
)

# --- Compute per-winner coverage threshold (15%) based on TOTAL ---
def compute_total_threshold(group, threshold=0):
    total = group["num_records"].sum()
    group = group.copy()
    group["pct"] = group["num_records"] / total

    selected = group[group["pct"] >= threshold].sort_values("pct", ascending=False)

    return pd.Series({
        "countries_over_15pct_list": selected["exporter_country"].tolist(),
        "countries_over_15pct_pct_list": selected["pct"].round(1).tolist(),  
        "countries_over_15pct_records_list": selected["num_records"].tolist(),
        "countries_over_15pct_count": len(selected),
        "total_countries": group["exporter_country"].nunique(),
    })


coverage_total = (
    winner_country_totals.groupby("winner_name")
    .apply(compute_total_threshold, threshold=0.1)
    .reset_index()
)

# Total lots per winner
total_lots_by_winner = (
    df_ted.groupby("winner_name")
          .size()
          .reset_index(name="total_winned_lots")
)

# CPV summary
lots_per_cpv = (
    df.groupby(["winner_name", "main_classification"])
      .size()
      .reset_index(name="lots_per_cpv")
)

lots_per_cpv_summary = (
    lots_per_cpv.groupby("winner_name")
    .apply(lambda g: dict(zip(g["main_classification"], g["lots_per_cpv"])))
    .reset_index(name="lots_per_cpv_dict")
)


lots_by_buyer_type = (
    df.groupby(["winner_name", "buyer_type"])
      .size()
      .reset_index(name="lots_per_group")
)

lots_by_buyer_summary = (
    lots_by_buyer_type.groupby("winner_name")
    .apply(lambda g: dict(zip(g["buyer_type"], g["lots_per_group"])))
    .reset_index(name="lots_per_buyer_group_dict")
)

# Records per winner-year
records_by_year = (
    full_exporter_stats.groupby(["winner_name", "year"])["num_records"]
    .sum()
    .reset_index()
)

records_by_year_list = (
    records_by_year.groupby("winner_name")
    .agg(
        years=("year", lambda x: sorted(x.unique())),
        records_by_year_list=("num_records", list)
    )
    .reset_index()
)


In [25]:
# Extract TED year
df_ted["ted_year"] = df_ted["publication_date"].dt.year

# Lots per winner-year from TED
lots_by_year = (
    df_ted.groupby(["winner_name", "ted_year"])
          .size()
          .reset_index(name="lots_in_year")
)

# Convert to compact list form
lots_by_year_list = (
    lots_by_year.groupby("winner_name")
                .agg(
                    ted_years=("ted_year", lambda x: sorted(x.unique())),
                    lots_by_year_list=("lots_in_year", list)
                )
                .reset_index()
)

# Final combined table
final_df = (
    coverage_total  # already winner-level totals
    .rename(columns={
        "total_records": "trade_total_records",
        "total_countries": "trade_total_countries",
        "countries_over_15pct_list": "trade_countries_over_15pct_list",
        "countries_over_15pct_pct_list": "trade_countries_over_15pct_pct_list",
        "countries_over_15pct_records_list": "trade_countries_over_15pct_records_list",
        "countries_over_15pct_count": "trade_countries_over_15pct_count"
    })
    .merge(records_by_year_list.rename(columns={
        "years": "trade_years",
        "records_by_year_list": "trade_records_by_year_list"
    }), on="winner_name", how="left")
    .merge(total_lots_by_winner.rename(columns={
        "total_lots": "ted_total_lots"
    }), on="winner_name", how="left")
    .merge(lots_per_cpv_summary.rename(columns={
        "lots_per_cpv_dict": "ted_lots_per_cpv_dict"
    }), on="winner_name", how="left")
    .merge(lots_by_buyer_summary.rename(columns={
        "lots_per_buyer_group_dict": "ted_lots_per_buyer_group_dict"
    }), on="winner_name", how="left")
    .merge(lots_by_year_list, on="winner_name", how="left")
)

In [26]:
final_df.total_winned_lots.sum()

np.int64(129)

In [27]:
final_df

,winner_name,trade_countries_over_15pct_list,trade_countries_over_15pct_pct_list,trade_countries_over_15pct_records_list,trade_countries_over_15pct_count,trade_total_countries,trade_years,trade_records_by_year_list,total_winned_lots,ted_lots_per_cpv_dict,ted_lots_per_buyer_group_dict,ted_years,lots_by_year_list
0,Arbeitsschutz RheinRuhr Service GmbH,[Sri Lanka],[1.0],[7],1,1,"[2024, 2025]","[3, 4]",3,"{'Clothing': 2, 'Miscellaneous services': 3, '...",{'cities': 7},"[2024, 2025]","[1, 2]"
1,Berendsen GmbH,[Pakistan],[1.0],[4],1,1,[2019],[4],2,"{18300000: 2, 39518000: 2}",{'Hospitals': 4},[2019],[2]
2,Bierbaum-Proenen GmbH & Co. KG,[Pakistan],[1.0],[28],1,1,[2025],[28],3,{'T-shirts and shirts': 28},{'cities': 28},[2025],[3]
3,Bonowi Hart Armour GmbH,[Pakistan],[1.0],[3],1,1,[2025],[3],1,{'Police equipment': 3},{'Police': 3},[2025],[1]
4,COP Vertriebs-GmbH,[Pakistan],[1.0],[7],1,1,[2022],[7],1,{35200000: 7},{'Police': 7},[2022],[1]
5,Carl Roth GmbH + Co.KG,[India],[1.0],[167],1,1,[2024],[167],1,{'Chemical products': 167},{'cities': 167},[2024],[1]
6,Comazo,"[India, Indonesia]","[0.8, 0.1]","[910, 154]",2,4,"[2023, 2024, 2025]","[374, 456, 248]",3,"{'Special workwear': 374, 'Underwear': 248, 'V...","{'Hospitals': 456, 'Police': 374, 'cities': 248}","[2023, 2024, 2025]","[1, 1, 1]"
7,Comazo GmbH & Co. KG,[India],[0.9],[590],1,4,"[2024, 2025]","[411, 248]",2,"{'T-shirts': 411, 'Underwear': 248}","{'Police': 411, 'cities': 248}","[2024, 2025]","[1, 1]"
8,Delta-Vertrieb GmbH,[Sri Lanka],[1.0],[19],1,1,[2024],[19],1,{'Protective gear': 19},{'cities': 19},[2024],[1]
9,ELVE AG,[Bangladesh],[1.0],[2],1,1,[2024],[2],1,{'T-shirts': 2},{'Police': 2},[2024],[1]


In [28]:
cpv_map = {
    18000000: "General textiles / clothing accessories",
    18100000: "Occupational clothing",
    18200000: "Outerwear",
    18300000: "Garments",
    35200000: "Police equipment",
    39518000: "Hospital textile articles"
}


In [29]:
def map_cpv(x):
    # already descriptive text → leave unchanged
    if isinstance(x, str) and not x.isdigit():
        return x

    # convert numeric strings → int
    try:
        key = int(x)
    except:
        return x

    # return mapped value if exists
    return cpv_map.get(key, x)


In [30]:
df_ted["main_classification"] = df_ted["main_classification"].apply(map_cpv)


In [31]:
# TED: extract classifications per winner
ted_classification = (
    df_ted.groupby("winner_name")["main_classification"]
          .apply(lambda x: sorted(set([str(i) for i in x if pd.notna(i)])))
          .reset_index(name="main_classification_list")
)


In [32]:
# TRADE: extract HS codes per winner based on matched 6-month records
trade_hs = (
    df_6m.groupby("winner_name")["hs_code"]
         .apply(lambda x: sorted(set([str(i) for i in x if pd.notna(i)])))
         .reset_index(name="hs_code_list")
)


In [33]:
final_df = (
    final_df
    .merge(ted_classification, on="winner_name", how="left")
    .merge(trade_hs, on="winner_name", how="left")
)


In [34]:
final_df = final_df.rename(columns={"main_classification_list": "ted_main_classification_list"})
final_df = final_df.rename(columns={"hs_code_list": "trade_hs_code_list"})

# Convert to list safely (some entries may already be lists or dict keys)
def to_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return [x]

final_df["ted_main_classification_list"] = final_df["ted_main_classification_list"].apply(to_list)

# Remove winners containing unwanted textile types
unwanted = ["Chemical products", "Presents and rewards"]

final_filtered = final_df[
    ~final_df["ted_main_classification_list"].apply(
        lambda lst: any(item in lst for item in unwanted)
    )
].copy()

display(final_filtered)


,winner_name,trade_countries_over_15pct_list,trade_countries_over_15pct_pct_list,trade_countries_over_15pct_records_list,trade_countries_over_15pct_count,trade_total_countries,trade_years,trade_records_by_year_list,total_winned_lots,ted_lots_per_cpv_dict,ted_lots_per_buyer_group_dict,ted_years,lots_by_year_list,ted_main_classification_list,trade_hs_code_list
0,Arbeitsschutz RheinRuhr Service GmbH,[Sri Lanka],[1.0],[7],1,1,"[2024, 2025]","[3, 4]",3,"{'Clothing': 2, 'Miscellaneous services': 3, '...",{'cities': 7},"[2024, 2025]","[1, 2]","[Clothing, Miscellaneous services, Washing and...",[61161000]
1,Berendsen GmbH,[Pakistan],[1.0],[4],1,1,[2019],[4],2,"{18300000: 2, 39518000: 2}",{'Hospitals': 4},[2019],[2],"[Garments, Hospital textile articles]","[63026010, 63026090]"
2,Bierbaum-Proenen GmbH & Co. KG,[Pakistan],[1.0],[28],1,1,[2025],[28],3,{'T-shirts and shirts': 28},{'cities': 28},[2025],[3],[T-shirts and shirts],"[62019000, 62033300, 62034200, 62034300, 62034..."
3,Bonowi Hart Armour GmbH,[Pakistan],[1.0],[3],1,1,[2025],[3],1,{'Police equipment': 3},{'Police': 3},[2025],[1],[Police equipment],"[62034900, 62072900, 62171000]"
4,COP Vertriebs-GmbH,[Pakistan],[1.0],[7],1,1,[2022],[7],1,{35200000: 7},{'Police': 7},[2022],[1],[Police equipment],"[62031200, 62032900, 62033300, 62034900, 62079..."
6,Comazo,"[India, Indonesia]","[0.8, 0.1]","[910, 154]",2,4,"[2023, 2024, 2025]","[374, 456, 248]",3,"{'Special workwear': 374, 'Underwear': 248, 'V...","{'Hospitals': 456, 'Police': 374, 'cities': 248}","[2023, 2024, 2025]","[1, 1, 1]","[Special workwear, Underwear, Visors]","[39269069, 40169990, 42022220, 48191010, 48196..."
7,Comazo GmbH & Co. KG,[India],[0.9],[590],1,4,"[2024, 2025]","[411, 248]",2,"{'T-shirts': 411, 'Underwear': 248}","{'Police': 411, 'cities': 248}","[2024, 2025]","[1, 1]","[T-shirts, Underwear]","[39269069, 40169990, 42022220, 48191010, 48196..."
8,Delta-Vertrieb GmbH,[Sri Lanka],[1.0],[19],1,1,[2024],[19],1,{'Protective gear': 19},{'cities': 19},[2024],[1],[Protective gear],"[61161000, 64041900]"
9,ELVE AG,[Bangladesh],[1.0],[2],1,1,[2024],[2],1,{'T-shirts': 2},{'Police': 2},[2024],[1],[T-shirts],"[62059000, 62069000]"
10,Elis GmbH,[Kazakhstan],[1.0],[52],1,1,[2025],[52],11,{'Washing and dry-cleaning services': 52},{'Hospitals': 52},[2025],[11],[Washing and dry-cleaning services],"[3926909709, 4202990000, 6307909800, 9032890000]"


In [35]:
final_df.total_winned_lots.sum()

np.int64(129)

In [36]:
df_plot = []

for _, row in final_filtered.iterrows():
    textile_types = row.get("ted_main_classification_list", [])
    countries = row.get("trade_countries_over_15pct_list", [])
    buyer_groups_dict = row.get("ted_lots_per_buyer_group_dict", {})
    lots = row.get("total_winned_lots", 0)

    # Skip empty rows
    if not isinstance(textile_types, list) or not isinstance(countries, list):
        continue
    if len(textile_types) == 0 or len(countries) == 0:
        continue

    # Expand buyer groups into list form
    if isinstance(buyer_groups_dict, dict):
        buyer_groups = list(buyer_groups_dict.keys())
    else:
        buyer_groups = []

    # If no buyer groups, mark as "unknown"
    if len(buyer_groups) == 0:
        buyer_groups = ["unknown"]

    # Expand combinations: textile × country × buyer_group
    for t in textile_types:
        for c in countries:
            for b in buyer_groups:
                df_plot.append({
                    "country": c,
                    "textile_type": t,
                    "buyer_type": b,
                    "lot_weight": lots
                })

df_plot = pd.DataFrame(df_plot)


In [37]:
# Darker color palette
dark_palette = [
    "#C75D2C", "#972484", "#517fd6", "#0a5728",
    "#0f766e", "#1d4ed8", "#92822ada", "#92400e",
    "#b91c1c", "#065f46", "#78350f", "#312e81"
]


In [38]:
plot_group_map = {

    "T-shirts": "T-shirts and shirts",
    "Socks": "Socks and Underwear",
    "Underwear": "Socks and Underwear",
    "Hospital textile articles": "Hospital linen"
}

In [39]:
exclude_types = [
    "Hard hats",
    "Protective footwear",
    "Footwear",
    "Garments",
    "Trousers",
    "Miscellaneous services",
    "Services",
    "Visors",
    "Security",
]


In [40]:
df_plot = df_plot[~df_plot["textile_type"].astype(str).isin(exclude_types)].copy()

In [41]:
df_plot["textile_type"] = df_plot["textile_type"].replace(plot_group_map)


In [42]:
df_plot["textile_type"].value_counts()

textile_type
Occupational clothing                      15
Socks and Underwear                        11
Weatherproof clothing                      11
Clothing                                   10
Police equipment                            9
Outerwear                                   9
Special workwear                            6
Hospital linen                              5
General textiles / clothing accessories     5
T-shirts and shirts                         4
Washing and dry-cleaning services           3
Protective gear                             1
Individual equipment                        1
Disposable gloves                           1
Name: count, dtype: int64

In [43]:

# Aggregate after merge
df_grouped = (
    df_plot.groupby(["country", "textile_type"])["lot_weight"]
           .sum()
           .reset_index()
)

df_pivot = df_grouped.pivot(
    index="country",
    columns="textile_type",
    values="lot_weight"
).fillna(0)


fig = px.bar(
    df_pivot,
    x=df_pivot.index,
    y=df_pivot.columns,
    title="Countries Supplying Each Textile Type (Weighted by Total Lots Won)",
    labels={"value": "Total Weighted Lots", "country": "Country"},
)

fig.update_traces(
    textangle=0,              
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(size=16)
)
fig.update_layout(
    barmode="stack",
    xaxis_title="Exporter Country",
    yaxis_title="Total Weighted Lots",
    legend_title="Textile Type",
    width=1300,
    height=750,
    template="plotly_white",
    font=dict(size=13)
)

fig.show()


In [44]:
colors = px.colors.qualitative.Safe_r

In [47]:
df_grouped = (
    df_plot.groupby(["textile_type", "country"])["lot_weight"]
    .sum()
    .reset_index()
)

df_pivot = df_grouped.pivot(
    index="textile_type",
    columns="country",
    values="lot_weight"
).fillna(0)

# Light professional palette

fig = px.bar(
    df_pivot,
    x=df_pivot.index,
    y=df_pivot.columns,
    title="Exporting Countries per Textile Type (Weighted by Total Lots Won)",
    labels={"value": "Total Weighted Lots", "textile_type": "Textile Type"},
    color_discrete_sequence=colors,
    text_auto=True
)

fig.update_traces(
    textangle=0,              
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(size=16)
)

fig.update_layout(
    barmode="stack",
    xaxis_title="Textile Type",
    yaxis_title="Total Weighted Lots",
    legend_title="Country",
    width=1400,
    height=750,
    
    # 🛠️ FIXED: Use title=dict(font=dict(...)) instead of titlefont
    xaxis=dict(
        tickfont=dict(size=15), 
        title=dict(font=dict(size=20))
    ),
    yaxis=dict(
        tickfont=dict(size=15), 
        title=dict(font=dict(size=20))
    ),
    
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(color="#222", size=13),
    margin=dict(l=40, r=40, t=90, b=120)
)

# Rotate x-axis labels for readability
fig.update_xaxes(tickangle=45)

fig.show()

In [49]:
df_buyer = df_plot[df_plot["buyer_type"].notna()].copy()

df_grouped_buyer = (
    df_buyer.groupby(["country", "buyer_type"])["lot_weight"]
            .sum()
            .reset_index()
)

df_pivot_buyer = df_grouped_buyer.pivot(
    index="country",
    columns="buyer_type",
    values="lot_weight"
).fillna(0)

fig_buyer = px.bar(
    df_pivot_buyer,
    x=df_pivot_buyer.index,
    y=df_pivot_buyer.columns,
    title="Public Institutions by Exporting Country (Weighted by Total Lots Won)",
    labels={"value": "Weighted Lots", "country": "Exporter Country"},
    color_discrete_sequence=dark_palette,
    text_auto=True
)

fig_buyer.update_traces(
    textangle=0,                 # horizontal
    textposition="inside",       # inside the bar
    insidetextanchor="middle"    ,    # centered inside segment
    textfont=dict(size=16)
)

fig.update_layout(
    barmode="stack",
    xaxis_title="Textile Type",
    yaxis_title="Total Weighted Lots",
    legend_title="Country",
    width=1400,
    height=750,
    
    # 🛠️ FIXED: Use title=dict(font=dict(...)) instead of titlefont
    xaxis=dict(
        tickfont=dict(size=15), 
        title=dict(font=dict(size=20))
    ),
    yaxis=dict(
        tickfont=dict(size=15), 
        title=dict(font=dict(size=20))
    ),
    
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(color="#222", size=13),
    margin=dict(l=40, r=40, t=90, b=120)
)

fig_buyer.show()
